In [8]:
SEED = 42

In [2]:
from pathlib import Path
from helpers.data.ticker_loader import load_tickers

TICKERS_FILE = Path("tickers.txt")
if not TICKERS_FILE.exists():
    TICKERS_FILE = Path("NN_Trading_project/tickers.txt")

# None        → all tickers
# ["AAPL", …] → explicit list
# 42          → random sample of 42 (seeded by SEED)
TICKER_SUBSET = 3

TICKERS = load_tickers(TICKERS_FILE, subset=TICKER_SUBSET, seed=SEED)
print(f"Using {len(TICKERS)} tickers")


Using 3 tickers


In [10]:
import pandas as pd
import datetime

# ============================================================
# DATE SETTINGS
# ============================================================

# --- Date Range ---
INTERVAL   = "1d"
START_DATE = pd.Timestamp("2025-01-01")           # inclusive
END_DATE   = pd.Timestamp(datetime.date.today())  # inclusive

# --- Train / Val / Test Split ---
# Train: Date <= TRAIN_END_DATE
# Val:   (TRAIN_END_DATE, VAL_END_DATE]
# Test:  Date > VAL_END_DATE
TRAIN_END_DATE = pd.Timestamp("2025-11-30")
VAL_END_DATE   = pd.Timestamp("2026-01-20")

# --- Exclusion Window (applied during featurize) ---
EXCLUDE_START_DATE = "2018-09-01"  # inclusive
EXCLUDE_END_DATE   = "2023-06-30"  # inclusive


# --- Exclusion Window (Features) ---
REBUILD_FEATURE_CACHE = True  # set True to recompute when features/WINDOW change


In [11]:
# ============================================================
# HYPERPARAMETERS - MODEL & TRAINING
# ============================================================

# --- Trading / Labeling ---
# Buy at market OPEN using prior day's OHLCV + current day's OPEN only.
HORIZON_BARS     = 0         # look-ahead bars for profit hit
PROFIT_THRESHOLD = 1 / 100   # 1% profit target vs current day's OPEN
STOP_LOSS        = -1 / 100  # -2% stop loss vs current day's OPEN
WINDOW           = 30        # look-back window for features

# --- Training ---
MAX_EPOCHS    = 50
PATIENCE      = 10   # early-stopping patience

BUY_THRESHOLD = 0.5  # minimum probability to classify as BUY

# --- Plotting ---
PLOT_GRAPH1_EPOCH_INTERVAL = 1
PLOT_GRAPH2_BATCH_INTERVAL = 5
PLOT_GRAPH3_EPOCH_INTERVAL = 1

# --- Optimizer ---
BATCH_SIZE   = 64

# --- Model Architecture ---
HIDDEN_SIZES = [64]  # e.g. [1024, 512] or [256, 128]

# --- Data / DataLoader ---
SPLIT_FRAC  = 0.85  # train fraction (time-based)
NUM_WORKERS = 16    # DataLoader workers (0 for debugging)


In [5]:
from pathlib import Path
from helpers.data.date_config_manager import check_and_refresh_date_config

_current_config = {
    "INTERVAL":           str(INTERVAL),
    "START_DATE":         str(START_DATE.date()),
    "END_DATE":           str(END_DATE.date()),
    "TRAIN_END_DATE":     str(TRAIN_END_DATE.date()),
    "VAL_END_DATE":       str(VAL_END_DATE.date()),
    "EXCLUDE_START_DATE": str(EXCLUDE_START_DATE),
    "EXCLUDE_END_DATE":   str(EXCLUDE_END_DATE),
    "TICKER_SUBSET":      str(TICKER_SUBSET),
}

check_and_refresh_date_config(
    current_config = _current_config,
    config_path    = Path.cwd() / "date_config.txt",
    stocks_dir     = Path.cwd() / "dataset" / "stocks",
)


Date config changed — clearing cached CSVs for a fresh download.
  Previous config:
    INTERVAL: 1d
    START_DATE: 2025-01-01
    END_DATE: 2026-04-03
    TRAIN_END_DATE: 2025-11-30
    VAL_END_DATE: 2026-01-20
    EXCLUDE_START_DATE: 2018-09-01
    EXCLUDE_END_DATE: 2023-06-30
    TICKER_SUBSET: 42 <-- CHANGED
  Deleted: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks
  Updated: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/date_config.txt


True

In [6]:
# Step 1: download NASDAQ data into the *dataset* directory using yfinance (DAILY)
from pathlib import Path
import pandas as pd
from helpers.data.data_downloader import download_tickers

# Root directory where yfinance CSVs will be stored

data_root = Path.cwd() / "dataset"
stocks_dir = data_root / "stocks"

# call helper
_summary = download_tickers(
    tickers= TICKERS,
    start=   START_DATE,
    end=     END_DATE,
    interval=INTERVAL,
    out_dir= stocks_dir,
)


 OK   :: AFRM rows=313 | downloaded 1/3
 OK   :: BSX rows=313 | downloaded 2/3
 OK   :: SRPT rows=313 | downloaded 3/3
Path to dataset files: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset
Individual ticker CSVs are in: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks
Total number of rows across all CSVs on disk: 942
Newly downloaded rows in this run: 939
Downloaded tickers in this run: 3/3

All requested tickers that were downloaded in this run succeeded.


# Building Features + Labels

In [7]:
import pickle
import shutil
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from helpers.feature.feature_builder import precompute_and_cache, FEATURE_COLS
from helpers.data.dataset import StockDatasetSafe, is_cache_valid

# ── Paths ────────────────────────────────────────────────────────────────────
root       = Path.cwd() / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files     = sorted(stocks_dir.glob("*.csv"))
cache_dir = Path.cwd() / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

# ── Clear old cache if requested ─────────────────────────────────────────────
if REBUILD_FEATURE_CACHE:
    stale = [p for p in Path.cwd().glob(".feature*") if p.exists()]
    for p in stale:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        else:
            p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    if not stale:
        print("[Cache] No stale .feature* paths found.")
    time.sleep(1)

# ── Build or load cache ───────────────────────────────────────────────────────
# Validates that scaler, index, AND every referenced .npz file all exist.
cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete cache detected — wiping cache dir.")
    shutil.rmtree(cache_dir)

# Ensure cache_dir exists before building or loading
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files           = files,
        window          = WINDOW,
        cache_dir       = cache_dir,
        scaler_path     = scaler_path,
        index_path      = index_path,
        horizon_bars    = HORIZON_BARS,
        train_end_date  = TRAIN_END_DATE,
        val_end_date    = VAL_END_DATE,
        profit_threshold= PROFIT_THRESHOLD,
        stop_loss       = STOP_LOSS,
        exclude_start   = EXCLUDE_START_DATE,
        exclude_end     = EXCLUDE_END_DATE,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pin    = torch.cuda.is_available()
_kwargs = dict(
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    pin_memory  = _pin,
    persistent_workers = (NUM_WORKERS > 0),
    prefetch_factor    = 2 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

# ── Sanity check ──────────────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Test  samples : {len(test_ds):,}")
print(f"X batch : {xb.shape}  {xb.dtype}")
print(f"y batch : {yb.shape}  {yb.dtype}")
print(f"Feature count : {len(FEATURE_COLS)} base + {WINDOW-1} lags = {len(FEATURE_COLS) * WINDOW}")

[Cache] Removed: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/.feature_cache_forward_return_w30
[Cache] Precomputing features (leakage-safe splits)...
[Cache] (1/3) AFRM.csv
[Cache] (2/3) BSX.csv
[Cache] (3/3) SRPT.csv
[Cache] Done — used=3, skipped=0
Train samples : 378
Val   samples : 102
Test  samples : 153
X batch : torch.Size([64, 1320])  torch.float32
y batch : torch.Size([64])  torch.float32
Feature count : 44 base + 29 lags = 1320


# XGBoost

In [8]:

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import log_loss

from helpers.evaluation import buy_metrics, predict_probs_booster

# ── Helpers ───────────────────────────────────────────────────────────────────

def loader_to_numpy(loader):
    """Flatten a torch DataLoader into (X, y) numpy arrays."""
    Xs, ys = [], []
    for xb, yb in loader:
        Xs.append(xb.numpy())
        ys.append(yb.numpy())
    return np.concatenate(Xs), np.concatenate(ys)


def logloss_curve(booster: xgb.Booster, X: np.ndarray, y: np.ndarray, iters: np.ndarray) -> np.ndarray:
    """Compute log-loss at each iteration count (post-training, for test curve)."""
    dmat = xgb.DMatrix(X)
    y = y.astype(np.int64)
    out = []
    for k in iters:
        try:
            p = booster.predict(dmat, iteration_range=(0, int(k)))
        except TypeError:
            p = booster.predict(dmat, ntree_limit=int(k))
        out.append(log_loss(y, p))
    return np.asarray(out, dtype=np.float64)


# ── 1. Build arrays from loaders ──────────────────────────────────────────────

X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

num_pos = float((y_train == 1).sum())
num_neg = float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)

print(f"Train {X_train.shape}  pos={int(num_pos)}  neg={int(num_neg)}")
print(f"Val   {X_val.shape}    pos={int((y_val==1).sum())}  neg={int((y_val==0).sum())}")
print(f"Test  {X_test.shape}   pos={int((y_test==1).sum())}  neg={int((y_test==0).sum())}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

# ── 2. Define params ──────────────────────────────────────────────────────────

NUM_BOOST_ROUND       = 10000
EARLY_STOPPING_ROUNDS = max(5, PATIENCE * 5)

params = {
    "max_depth":        3,
    "eta":              0.01,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "alpha":            3.0,
    "lambda":           2.0,
    "scale_pos_weight": scale_pos_weight,
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "tree_method":      "hist",
}

# ── 3. Train ──────────────────────────────────────────────────────────────────

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

evals_result = {}
print(f"Training: max_rounds={NUM_BOOST_ROUND}, early_stop={EARLY_STOPPING_ROUNDS}")
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    evals_result=evals_result,
    verbose_eval=100,
)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else NUM_BOOST_ROUND
print(f"\nBest iteration: {best_ntree}")

# ── 4. Final metrics ──────────────────────────────────────────────────────────

train_ll = np.asarray(evals_result["train"]["logloss"])
val_ll   = np.asarray(evals_result["val"]["logloss"])

probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)
probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
tr = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
vl = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)
te = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)

print(f"Train  logloss={log_loss(y_train,probs_train):.6f}  acc={tr['acc']:.2f}%  P(success|BUY)={tr['buy_success']:.2f}%")
print(f"Val    logloss={log_loss(y_val,  probs_val  ):.6f}  acc={vl['acc']:.2f}%  P(success|BUY)={vl['buy_success']:.2f}%")
print(f"Test   logloss={log_loss(y_test, probs_test ):.6f}  acc={te['acc']:.2f}%  P(success|BUY)={te['buy_success']:.2f}%")

# ── 5. Training curves ────────────────────────────────────────────────────────

iters_full = np.arange(1, len(train_ll) + 1)

# ── 6. Save model ────────────────────────────────────────────────────────────

bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("Saved → best_model_xgb.pkl")


Train (378, 1320)  pos=110  neg=268
Val   (102, 1320)    pos=26  neg=76
Test  (153, 1320)   pos=43  neg=110
scale_pos_weight = 2.4364
Training: max_rounds=10000, early_stop=50
[0]	train-logloss:0.69090	val-logloss:0.69277
[100]	train-logloss:0.52919	val-logloss:0.67283
[200]	train-logloss:0.42297	val-logloss:0.65894
[300]	train-logloss:0.34983	val-logloss:0.65160
[400]	train-logloss:0.29230	val-logloss:0.64370
[458]	train-logloss:0.26605	val-logloss:0.64508

Best iteration: 409
Train  logloss=0.288069  acc=100.00%  P(success|BUY)=100.00%
Val    logloss=0.642277  acc=62.75%  P(success|BUY)=22.73%
Test   logloss=0.666833  acc=60.78%  P(success|BUY)=30.23%
Saved → best_model_xgb.pkl


### Evaluate on Test Data 

In [9]:

from helpers.evaluation import evaluate_on_test_data

df_test_preds, df_test_summary = evaluate_on_test_data(
    booster=booster,
    ntree=best_ntree,
    X_test=X_test,
    y_test=y_test,
    threshold=BUY_THRESHOLD,
    index_path=index_path,
    stocks_dir=stocks_dir,
    save_csv="test_predictions_full.csv",
)


Saved test predictions -> test_predictions_full.csv
P(success | BUY): 30.23%  |  acc: 60.78%


# Optuna Hyperparameter Search (XGBoost)

In [ ]:
# ── Optuna: XGBoost Hyperparameter Search ─────────────────────────────────────

import numpy as np
import optuna
import xgboost as xgb
from sklearn.metrics import log_loss
import wandb

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── W&B Setup ─────────────────────────────────────────────────────────────────
wandb.login(key="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")

wandb.init(
    project="NN-Trading-Bot",
    # Name will be set after finding best params so it can include values
    name=None,
    config={
        "optuna_n_trials":    80,
        "optuna_early_stop":  max(10, PATIENCE * 5),
        "optuna_max_rounds":  3000,
        "buy_threshold":      BUY_THRESHOLD,
        "profit_threshold":   PROFIT_THRESHOLD,
        "stop_loss":          STOP_LOSS,
        "window":             WINDOW,
        "seed":               SEED,
    },
)

OPTUNA_N_TRIALS     = 80
OPTUNA_EARLY_STOP   = max(10, PATIENCE * 5)
OPTUNA_MAX_ROUNDS   = 3000          # max boosting rounds per trial

dtrain_opt = xgb.DMatrix(X_train, label=y_train)
dval_opt   = xgb.DMatrix(X_val,   label=y_val)
dtest_opt  = xgb.DMatrix(X_test,  label=y_test)


def objective(trial: optuna.Trial) -> float:
    """Minimise validation log-loss."""
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      "logloss",
        "tree_method":      "hist",
        "seed":             SEED,
        "scale_pos_weight": scale_pos_weight,
        # ── tuneable ────────────────────────────────────────────────────────
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "eta":              trial.suggest_float("eta", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "alpha":            trial.suggest_float("alpha", 0.0, 10.0),
        "lambda":           trial.suggest_float("lambda", 0.5, 10.0),
    }

    evals_res = {}
    bst = xgb.train(
        params=params,
        dtrain=dtrain_opt,
        num_boost_round=OPTUNA_MAX_ROUNDS,
        evals=[(dval_opt, "val")],
        early_stopping_rounds=OPTUNA_EARLY_STOP,
        evals_result=evals_res,
        verbose_eval=False,
    )

    best_iter   = int(bst.best_iteration + 1) if bst.best_iteration is not None else OPTUNA_MAX_ROUNDS
    val_logloss = evals_res["val"]["logloss"][best_iter - 1]

    trial.set_user_attr("best_ntree",   best_iter)
    trial.set_user_attr("val_logloss",  val_logloss)

    # ── Log trial to W&B ──────────────────────────────────────────────────
    wandb.log({
        "trial_number":     trial.number,
        "val_logloss":      val_logloss,
        "best_val_logloss": min(val_logloss, wandb.run.summary.get("best_val_logloss", float("inf"))),
        "best_ntree":       best_iter,
        **{k: v for k, v in params.items() if k not in ("objective", "eval_metric", "tree_method", "seed", "scale_pos_weight")},
    })

    return val_logloss


# ── Run the study ─────────────────────────────────────────────────────────────

study = optuna.create_study(direction="minimize",
                             study_name="xgb_hparam_search",
                             sampler=optuna.samplers.TPESampler(seed=SEED))

print(f"Starting Optuna search: {OPTUNA_N_TRIALS} trials …")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)

best_trial  = study.best_trial
best_params = best_trial.params
best_val_ll = best_trial.value
print(f"\nBest trial #{best_trial.number}  val_logloss={best_val_ll:.6f}")
print("Best params:", best_params)

# Set W&B run name to best hyperparameter values for easier identification
best_params_sorted = sorted(best_params.items())
best_param_name = "optuna_" + "____".join(f"{k}_{v}" for k, v in best_params_sorted)
if wandb.run is not None:
    wandb.run.name = best_param_name
    wandb.run.save()
    print(f"W&B run renamed to: {wandb.run.name}")

# ── Log best trial summary to W&B ─────────────────────────────────────────────
wandb.run.summary["best_trial_number"] = best_trial.number
wandb.run.summary["best_val_logloss"]  = best_val_ll
wandb.run.summary["best_params"]       = best_params

# ── Log Optuna visualisations to W&B ──────────────────────────────────────────
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_contour,
    plot_slice,
)

wandb.log({"optuna/optimization_history": wandb.Plotly(plot_optimization_history(study))})
wandb.log({"optuna/param_importances":    wandb.Plotly(plot_param_importances(study))})
wandb.log({"optuna/parallel_coordinate":  wandb.Plotly(plot_parallel_coordinate(study))})
wandb.log({"optuna/contour":              wandb.Plotly(plot_contour(study))})
wandb.log({"optuna/slice":                wandb.Plotly(plot_slice(study))})

# ── Retrain with best params on train+val ──────────────────────────────────────

final_params = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "tree_method":      "hist",
    "seed":             SEED,
    "scale_pos_weight": scale_pos_weight,
    **best_params,
}

# Use best_ntree from the winning trial (already tuned via early-stopping)
best_ntree_optuna = int(best_trial.user_attrs["best_ntree"])

dtrain_full = xgb.DMatrix(
    np.concatenate([X_train, X_val]),
    label=np.concatenate([y_train, y_val]),
)

print(f"\nRetraining on train+val for {best_ntree_optuna} rounds …")
booster = xgb.train(
    params=final_params,
    dtrain=dtrain_full,
    num_boost_round=best_ntree_optuna,
    verbose_eval=False,
)
best_ntree = best_ntree_optuna

# ── Evaluate on test ───────────────────────────────────────────────────────────

probs_test_optuna = predict_probs_booster(booster, X_test, best_ntree)
te_opt = buy_metrics(y_test, probs_test_optuna, BUY_THRESHOLD)
test_ll_val = log_loss(y_test, probs_test_optuna)

probs_train_optuna = predict_probs_booster(booster, X_train, best_ntree)
tr_opt = buy_metrics(y_train, probs_train_optuna, BUY_THRESHOLD)

print(f"\nTrain  acc={tr_opt['acc']:.2f}%  P(success|BUY)={tr_opt['buy_success']:.2f}%")
print(f"Test   acc={te_opt['acc']:.2f}%  P(success|BUY)={te_opt['buy_success']:.2f}%  logloss={test_ll_val:.6f}")

# ── Log final test metrics to W&B ─────────────────────────────────────────────
wandb.run.summary["test_logloss"]      = test_ll_val
wandb.run.summary["test_accuracy"]     = te_opt["acc"]
wandb.run.summary["test_buy_success"]  = te_opt["buy_success"]
wandb.run.summary["train_accuracy"]    = tr_opt["acc"]
wandb.run.summary["train_buy_success"] = tr_opt["buy_success"]

# ── Plots ──────────────────────────────────────────────────────────────────────

# 1. Optimisation history

# ── Save retrained model ──────────────────────────────────────────────────────
import joblib
bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("\nSaved → best_model_xgb.pkl")
print(f"Use BUY_THRESHOLD = {BUY_THRESHOLD} (unchanged; tune separately if needed)")

# ── Finish W&B run ────────────────────────────────────────────────────────────
wandb.finish()
print("W&B run finished — check your dashboard for plots.")


/scratch/temp_/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/adma00003/.netrc
wandb: Currently logged in as: adhammahgoub (adhammahgoub-uds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/temp_/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/adma00003/.netrc
wandb: Currently logged in as: adhammahgoub (adhammahgoub-uds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/temp_/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/adma00003/.netrc
wandb: Currently logged in as: adhammahgoub (adhammahgoub-uds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/scratch/temp_/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/adma00003/.netrc
wandb: Currently logged in as: adhammahgoub (adhammahgoub-uds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting Optuna search: 80 trials …


Best trial: 0. Best value: 0.616045:   8%|▊         | 6/80 [00:14<03:24,  2.77s/it]

# Eval Data Analysis

In [ ]:

# ── Val Data Analysis: split-agnostic helper ──────────────────────────────────

import wandb
from helpers.evaluation import evaluate_split_from_artifacts

SELECTED_THRESHOLD = 0.9

val_preds, daily, val_summary = evaluate_split_from_artifacts(
    split="val",
    threshold=SELECTED_THRESHOLD,
    index_path=index_path,
    scaler_path=scaler_path,
    model_path="best_model_xgb.pkl",
    verbose=True,
)

display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "val_analysis/threshold":    SELECTED_THRESHOLD,
        "val_analysis/total_trades": int(val_summary["total_trades"]),
        "val_analysis/pct_success":  float(val_summary["pct_success"]),
        "val_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })


# Test Data Analysis

In [ ]:

# ── Test Data Analysis: split-agnostic helper ─────────────────────────────────

import wandb
from helpers.evaluation import evaluate_split_from_artifacts

SELECTED_THRESHOLD = 0.9

test_preds, daily, test_summary = evaluate_split_from_artifacts(
    split="test",
    threshold=SELECTED_THRESHOLD,
    index_path=index_path,
    scaler_path=scaler_path,
    model_path="best_model_xgb.pkl",
    verbose=True,
)

display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

if wandb.run is not None:
    wandb.log({
        "test_analysis/threshold":    SELECTED_THRESHOLD,
        "test_analysis/total_trades": int(test_summary["total_trades"]),
        "test_analysis/pct_success":  float(test_summary["pct_success"]),
        "test_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
    })


In [ ]:
# don't execute the next cells
raise KeyboardInterrupt